# forward 3d full

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/elma16/beamax/blob/main/examples/forward/forward_3d_full.ipynb)

> Select **Runtime → Change runtime type → GPU or TPU** in Colab to demonstrate the hardware-acceleration story.


In [ ]:
# Install beamax for Google Colab. Safe to skip when running locally.
%%capture
%pip install --quiet "beamax[kwave] @ git+https://github.com/elma16/beamax.git"


In [ ]:
#!/usr/bin/env python
# coding: utf-8

"""
3D forward PAT experiments: k-Wave vs MSGB vs Hybrid.

Experiments:
  - Two Gaussian wavepackets (homogeneous c, heterogeneous c)
  - Point source (homogeneous c, heterogeneous c)
  - OABreast volume (homogeneous c, heterogeneous c)

For each experiment we:
  1. Build domain, sensors, and time grid.
  2. Run k-Wave forward solve.
  3. Run MSGB forward solve.
  4. Run Hybrid (LF/HF) forward solve.
  5. Compute relative L2, Linf, and energy errors vs k-Wave.
"""

import csv
import jax
import jax.numpy as jnp
import numpy as np

from pathlib import Path
from time import time

import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.patches as patches
import matplotlib.patheffects as pe
from mpl_toolkits.axes_grid1 import make_axes_locatable

from beamax import geometry, plotter, utils
from beamax.decomposition import DyadicDecomposition
from beamax.transforms import MSWPT
from beamax.gb import gb_solvers
from beamax.solvers import KWaveSolver, MSGBSolver, HybridSolver
from beamax.solvers.hybrid_solver_utils import get_indices_with_norm_less_than
from beamax.solvers import ShardingStrategy
from beamax.gb import SolverConfig

from kwave.options.simulation_execution_options import SimulationExecutionOptions
from kwave.options.simulation_options import SimulationOptions

# -----------------------------------------------------------------------------
# Global config & paths
# -----------------------------------------------------------------------------

ROOT_DIR = utils.detect_root()
PLOT_DIR = Path(ROOT_DIR / "plots")
DATA_DIR = Path(ROOT_DIR / "data")
PLOT_DIR.mkdir(exist_ok=True)
DATA_DIR.mkdir(exist_ok=True)

pltgb = plotter.PlotHelper()

jax.config.update("jax_enable_x64", True)
jax.config.update("jax_compilation_cache_dir", "/tmp/jax_cache")
jax.config.update("jax_persistent_cache_min_entry_size_bytes", -1)
jax.config.update("jax_persistent_cache_min_compile_time_secs", 0)
jax.config.update(
    "jax_persistent_cache_enable_xla_caches", "xla_gpu_per_fusion_autotune_cache_dir"
)

# Try to use thesis style if available
mpl_style = DATA_DIR / "thesis.mplstyle"
if mpl_style.exists():
    plt.style.use(mpl_style)

# Devices
is_cpu, is_gpu, is_tpu = utils.get_devices()
num_devices = len(jax.devices())
print(
    f"JAX devices: CPU={is_cpu}, GPU={is_gpu}, TPU={is_tpu}, num_devices={num_devices}"
)

# -----------------------------------------------------------------------------
# Error metrics (sensor space-time)
# -----------------------------------------------------------------------------


def relative_L2(u, ref):
    """Relative L2 error over (t, sensor_index)."""
    u_arr = np.asarray(u)
    r_arr = np.asarray(ref)
    diff = u_arr - r_arr
    num = np.linalg.norm(diff.ravel())
    den = np.linalg.norm(r_arr.ravel())
    return float(num / den)


def relative_Linf(u, ref):
    """Relative L∞ error over (t, sensor_index)."""
    u_arr = np.asarray(u)
    r_arr = np.asarray(ref)
    num = np.max(np.abs(u_arr - r_arr))
    den = np.max(np.abs(r_arr))
    return float(num / den)


def energy_error(u, ref):
    """Relative energy error: | ||u||_2 - ||ref||_2 | / ||ref||_2."""
    u_arr = np.asarray(u)
    r_arr = np.asarray(ref)
    eu = np.linalg.norm(u_arr.ravel())
    er = np.linalg.norm(r_arr.ravel())
    return float(abs(eu - er) / er)


# -----------------------------------------------------------------------------
# Domain & MSWPT setup (shared across experiments)
# -----------------------------------------------------------------------------

d = 3
N = (128,) * d
dx = (1e-4,) * d
periodic = (True,) * d
box_aspect_ratio = (1,) * d
num_levels = 2
num_boxes_levels = tuple(2 ** (i + 2) for i in range(num_levels))


# Base homogeneous c(x)
def c_hom(x):
    return 1500.0 + 0.0 * x[..., 0]


def c_het_gauss(x):
    """
    Heterogeneous c(x) with one fast and one slow Gaussian blob in 3D.
    """
    extent = tuple([N[i] * dx[i] for i in range(d)])
    background_speed = 1500.0

    center_fast = (0.4 * extent[0], 0.4 * extent[1], 0.4 * extent[2])
    width_fast = 0.05 * extent[0]
    amp_fast = 50.0

    center_slow = (0.6 * extent[0], 0.6 * extent[1], 0.6 * extent[2])
    width_slow = 0.08 * extent[0]
    amp_slow = 50.0

    fast_blob = jnp.exp(
        -(
            (x[..., 0] - center_fast[0]) ** 2
            + (x[..., 1] - center_fast[1]) ** 2
            + (x[..., 2] - center_fast[2]) ** 2
        )
        / (2 * width_fast**2)
    )
    slow_blob = jnp.exp(
        -(
            (x[..., 0] - center_slow[0]) ** 2
            + (x[..., 1] - center_slow[1]) ** 2
            + (x[..., 2] - center_slow[2]) ** 2
        )
        / (2 * width_slow**2)
    )
    return background_speed + amp_fast * fast_blob - amp_slow * slow_blob


# CFL for k-Wave-style timestep (3D)
cfl = float((jnp.sqrt(3) / 4).round(3))

# Dyadic decomposition and MSWPT
t0 = time()
dyadic_decomp = DyadicDecomposition(num_levels, N, num_boxes_levels, box_aspect_ratio)
redundancy = 2
windowing = "rectangular_mirror"
none_windowing = "none"

wpt = MSWPT(dyadic_decomp, redundancy, windowing)
wpt_none = MSWPT(dyadic_decomp, redundancy, none_windowing)
t1 = time()
print(f"Time to create MSWPT params: {t1 - t0:.3f}s")

# -----------------------------------------------------------------------------
# Initial condition constructors
# -----------------------------------------------------------------------------


def make_p0_two_wavepackets():
    from beamax import transforms

    KXY = dyadic_decomp.fourier_meshgrid

    boxhf = 44
    boxlf = 10

    khf = jnp.array([10, 12, 12])
    klf = jnp.array([10, 3, 3])
    kerft_hf = transforms.compute_frames(
        dyadic_decomp, boxhf, khf, KXY, redundancy, "none"
    )
    kerft_lf = transforms.compute_frames(
        dyadic_decomp, boxlf, klf, KXY, redundancy, "none"
    )

    p0 = utils.unitary_ifft(kerft_hf) + utils.unitary_ifft(kerft_lf)
    p0 = p0 / jnp.max(jnp.abs(p0))
    p0 = p0.T
    p0 = p0.real

    return p0


def make_p0_point_source():
    """Single point source at domain center."""
    p0 = jnp.zeros(N)
    p0 = p0.at[N[0] // 4, N[1] // 4, N[2] // 2].set(1.0)
    return p0


def make_p0_oabreast(dx_override=None):
    """
    Load a 3D OABreast phantom and resample to N.
    Requires the dataset to be present under DATA_DIR.
    """
    phantom_path = (
        DATA_DIR / "NumericalBreastPhantoms-selected" / "hdf5" / "Neg_07_Left.h5"
    )
    if not phantom_path.exists():
        raise FileNotFoundError(
            f"OABreast phantom not found at {phantom_path}. "
            "Please adjust path or skip this experiment."
        )

    spacing_mm = dx_override if dx_override is not None else dx
    p0, c_3d, meta3d = utils.load_oabreast_p0_c(
        phantom_path,
        dim="3d",
        axis_order="ZYX",
        source_spacing_mm=spacing_mm,
        target_shape=N,
        normalize_p0=True,
        label_to_sos={4: 1550.0},
    )
    return p0.real, c_3d


# -----------------------------------------------------------------------------
# Plotting helpers (3D-specific with MIP projections)
# -----------------------------------------------------------------------------


def mip(arr, axis):
    """Maximum intensity projection along given axis."""
    return np.max(arr, axis=axis)


def shared_norm(arr_list):
    """Shared normalization across arrays."""
    vmin = min(float(np.min(a)) for a in arr_list)
    vmax = max(float(np.max(a)) for a in arr_list)
    return mcolors.Normalize(vmin=vmin, vmax=vmax)


def symmetric_norm(arr_list):
    """Symmetric normalization for difference plots."""
    m = max(float(np.max(np.abs(a))) for a in arr_list)
    return mcolors.Normalize(vmin=-m, vmax=+m)


def add_colorbar(fig, ax, im, where="right", size="5%", pad=0.08):
    """Add colorbar to axis."""
    div = make_axes_locatable(ax)
    cax = div.append_axes(where, size=size, pad=pad)
    fig.colorbar(im, cax=cax)
    return cax


def plot_experiment_results(
    exp_name: str,
    p0: jnp.ndarray,
    p0_recon: jnp.ndarray,
    coeffs_array: jnp.ndarray,
    domain: geometry.Domain,
    sensors: geometry.Sensor,
    ts: jnp.ndarray,
    sensor_kw: np.ndarray,
    sensor_msgb: np.ndarray,
    sensor_hyb: np.ndarray,
    dyadic_decomp: DyadicDecomposition,
    cutoff_freq: float | None,
):
    """
    Produce per-experiment diagnostic figures for 3D experiments using MIP projections.
    """
    kw = np.asarray(sensor_kw)
    msgb = np.asarray(sensor_msgb)
    hyb = np.asarray(sensor_hyb) if sensor_hyb is not None else None
    hyb_arr = hyb if hyb is not None else msgb
    ts_arr = np.asarray(ts)

    p0_real = np.asarray(p0.real)
    recon_diff = np.asarray(p0_recon.real - p0_real)
    coeffs_mag = np.asarray(np.abs(coeffs_array))

    # MIP projections for p0, coefficients, and reconstruction error
    p0_mip = mip(p0_real, axis=2)  # XY projection
    coeffs_mip = mip(coeffs_mag, axis=2)
    recon_diff_mip = mip(np.abs(recon_diff), axis=2)
    c_arr = np.asarray(domain.c_fn(domain.grid))
    c_mip = mip(c_arr, axis=2)

    # Reshape sensor data: (Nt, Nx, Ny) with MIP over sensor plane
    Nt = len(ts)
    # Sensor data is flattened, reshape based on sensor mask shape
    sensor_shape = (Nt, N[0], N[1])  # planar sensor at z=0

    # MIP over z-axis of sensor plane (taking max across XY for visualization)
    kw_reshaped = kw.reshape(sensor_shape) if kw.size == np.prod(sensor_shape) else kw
    msgb_reshaped = (
        msgb.reshape(sensor_shape) if msgb.size == np.prod(sensor_shape) else msgb
    )
    hyb_reshaped = (
        hyb_arr.reshape(sensor_shape)
        if hyb_arr.size == np.prod(sensor_shape)
        else hyb_arr
    )

    # Take MIP along spatial axis for visualization
    kw_mip = (
        mip(np.asarray(kw_reshaped), axis=2) if kw_reshaped.ndim == 3 else kw_reshaped
    )
    msgb_mip = (
        mip(np.asarray(msgb_reshaped), axis=2)
        if msgb_reshaped.ndim == 3
        else msgb_reshaped
    )
    hyb_mip = (
        mip(np.asarray(hyb_reshaped), axis=2)
        if hyb_reshaped.ndim == 3
        else hyb_reshaped
    )

    # Shared color scaling
    sensor_vals = [kw_mip, msgb_mip, hyb_mip]
    s_min = float(min(arr.min() for arr in sensor_vals))
    s_max = float(max(arr.max() for arr in sensor_vals))
    s_norm = mcolors.Normalize(vmin=s_min, vmax=s_max)

    d_kw_msgb = kw_mip - msgb_mip
    d_kw_hyb = kw_mip - hyb_mip
    d_absmax = float(max(np.max(np.abs(d_kw_msgb)), np.max(np.abs(d_kw_hyb))))
    d_norm = mcolors.Normalize(vmin=-d_absmax, vmax=d_absmax)

    rd_max = float(np.max(np.abs(recon_diff_mip)))
    rd_norm = mcolors.Normalize(vmin=0, vmax=rd_max)

    extent_sensor = [
        0,
        domain.N[1] * domain.dx[1],
        float(jnp.max(ts)),
        float(jnp.min(ts)),
    ]
    extent_coeffs = [
        -domain.N[0] // 2,
        domain.N[0] // 2,
        -domain.N[1] // 2,
        domain.N[1] // 2,
    ]

    # --- Figure 1: c(x), p0, coeffs, recon diff (MIP projections) ---
    fig1 = plt.figure(figsize=(14, 3.5))
    gs1 = fig1.add_gridspec(nrows=1, ncols=4, wspace=0.5)

    ax_cmap = fig1.add_subplot(gs1[0, 0])
    im_c = ax_cmap.imshow(c_mip, origin="lower")
    ax_cmap.set_title(r"$\max_z\, c(\mathbf{x})$")
    ax_cmap.set_xticks([])
    ax_cmap.set_yticks([])
    div_c = make_axes_locatable(ax_cmap)
    cax_c = div_c.append_axes("right", size="5%", pad=0.1)
    fig1.colorbar(im_c, cax=cax_c)

    ax_p0 = fig1.add_subplot(gs1[0, 1])
    im_p0 = ax_p0.imshow(p0_mip, origin="lower", cmap="hot")
    ax_p0.set_title(r"$\max_z\, p_0(\mathbf{x})$")
    ax_p0.set_xticks([])
    ax_p0.set_yticks([])
    div_p0 = make_axes_locatable(ax_p0)
    cax_p0 = div_p0.append_axes("right", size="5%", pad=0.1)
    fig1.colorbar(im_p0, cax=cax_p0)

    ax_coeff = fig1.add_subplot(gs1[0, 2])
    im_coeff = ax_coeff.imshow(coeffs_mip, origin="lower", extent=extent_coeffs)
    # Draw dyadic boxes (2D projection)
    cumsum_boxes = jnp.r_[0, jnp.cumsum(dyadic_decomp.num_boxes_ndim)]
    box_lengths = dyadic_decomp.box_lengths
    colors = ["gray", "darkgray", "silver", "lightgray"]
    for level in range(dyadic_decomp.num_levels):
        start_idx = cumsum_boxes[level]
        end_idx = cumsum_boxes[level + 1]
        centers = dyadic_decomp.centres_ndim[start_idx:end_idx]
        box_length = box_lengths[level]
        box_width_x = box_length * dyadic_decomp.box_aspect_ratio[0]
        box_width_y = box_length * dyadic_decomp.box_aspect_ratio[1]
        color = colors[level % len(colors)]
        for center in centers:
            x_min = center[0] - box_width_x // 2
            x_max = center[0] + box_width_x // 2
            y_min = center[1] - box_width_y // 2
            y_max = center[1] + box_width_y // 2
            rect = patches.Rectangle(
                (x_min, y_min),
                x_max - x_min,
                y_max - y_min,
                linewidth=1.0,
                edgecolor=color,
                facecolor="none",
                linestyle=":",
                alpha=0.6,
            )
            ax_coeff.add_patch(rect)

    # If a cutoff is set, outline the selected low-frequency region.
    if cutoff_freq is not None:
        idx = get_indices_with_norm_less_than(dyadic_decomp.centres_ndim, cutoff_freq)
        if idx.size > 0:
            bounds_x, bounds_y = [], []
            for box_idx in idx:
                level = utils.find_level(dyadic_decomp, int(box_idx))
                center = dyadic_decomp.centres_ndim[int(box_idx)]
                box_length = box_lengths[level]
                box_width_x = box_length * dyadic_decomp.box_aspect_ratio[0]
                box_width_y = box_length * dyadic_decomp.box_aspect_ratio[1]
                bounds_x.extend(
                    [center[0] - box_width_x // 2, center[0] + box_width_x // 2]
                )
                bounds_y.extend(
                    [center[1] - box_width_y // 2, center[1] + box_width_y // 2]
                )
            x_min, x_max = min(bounds_x), max(bounds_x)
            y_min, y_max = min(bounds_y), max(bounds_y)
            rect = patches.Rectangle(
                (x_min, y_min),
                x_max - x_min,
                y_max - y_min,
                linewidth=2.0,
                edgecolor="red",
                facecolor="none",
                linestyle="--",
                zorder=5,
            )
            ax_coeff.add_patch(rect)

    ax_coeff.set_title(r"$\max_{k_z} |c_{\ell,j,k}|$")
    ax_coeff.set_xticks([])
    ax_coeff.set_yticks([])
    div_coeff = make_axes_locatable(ax_coeff)
    cax_coeff = div_coeff.append_axes("right", size="5%", pad=0.1)
    fig1.colorbar(im_coeff, cax=cax_coeff)

    ax_rdif = fig1.add_subplot(gs1[0, 3])
    im_rdif = ax_rdif.imshow(recon_diff_mip, origin="lower", norm=rd_norm, cmap="hot")
    ax_rdif.set_title(r"$\max_z |p_0^{\mathrm{recon}} - p_0|$")
    ax_rdif.set_xticks([])
    ax_rdif.set_yticks([])
    div_rdif = make_axes_locatable(ax_rdif)
    cax_rdif = div_rdif.append_axes("right", size="5%", pad=0.1)
    fig1.colorbar(im_rdif, cax=cax_rdif)

    fig1.tight_layout()
    fig1.savefig(
        PLOT_DIR / f"{exp_name}_fig1_initial.png", dpi=300, bbox_inches="tight"
    )
    plt.close(fig1)

    # --- Figure 2: sensor data and profile (MIP) ---
    fig2 = plt.figure(figsize=(10, 7))
    gs2 = fig2.add_gridspec(
        nrows=2, ncols=3, height_ratios=[1.0, 0.75], hspace=0.4, wspace=0.15
    )

    ax_kw = fig2.add_subplot(gs2[0, 0])
    ax_dmsgb = fig2.add_subplot(gs2[0, 1])
    ax_dhyb = fig2.add_subplot(gs2[0, 2])

    im_kw = ax_kw.imshow(kw_mip, extent=extent_sensor, aspect="auto", norm=s_norm)
    ax_dmsgb.imshow(
        d_kw_msgb, extent=extent_sensor, aspect="auto", norm=d_norm, cmap="RdBu_r"
    )
    im_dhyb = ax_dhyb.imshow(
        d_kw_hyb, extent=extent_sensor, aspect="auto", norm=d_norm, cmap="RdBu_r"
    )

    ax_kw.set_title(r"$\max_z\, p_t^{\mathrm{k\text{-}Wave}}$")
    ax_kw.set_xlabel("$y_{s}$")
    ax_kw.set_ylabel("t")
    ax_kw.set_xticks([])
    ax_kw.set_yticks([])

    ax_dmsgb.set_title(
        r"$\max_z\, (p_t^{\mathrm{k\text{-}Wave}} - p_t^{\mathrm{MSGB}})$"
    )
    ax_dhyb.set_title(
        r"$\max_z\, (p_t^{\mathrm{k\text{-}Wave}} - p_t^{\mathrm{Hybrid}})$"
    )
    for ax in (ax_dmsgb, ax_dhyb):
        ax.set_xticks([])
        ax.set_yticks([])

    divL = make_axes_locatable(ax_kw)
    caxL = divL.append_axes("left", size="7%", pad=0.22)
    fig2.colorbar(im_kw, cax=caxL)
    caxL.yaxis.set_ticks_position("left")
    caxL.yaxis.set_label_position("left")

    divR = make_axes_locatable(ax_dhyb)
    caxR = divR.append_axes("right", size="7%", pad=0.2)
    fig2.colorbar(im_dhyb, cax=caxR)

    Ny, Nx = kw_mip.shape
    xs = np.linspace(extent_sensor[0], extent_sensor[1], Nx)
    max_diff_per_sensor = np.max(np.abs(d_kw_msgb), axis=0)
    if Nx > 2:
        interior_max = np.argmax(max_diff_per_sensor[1:-1])
        idx_profile = int(interior_max + 1)
    else:
        idx_profile = int(np.argmax(max_diff_per_sensor))
    idx_profile = int(np.clip(idx_profile, 0, Nx - 1))
    profile_pos = xs[idx_profile] if Nx > 0 else 0.0
    trace_lw = 0.9
    trace_outline_lw = 1.6
    for ax in (ax_kw, ax_dmsgb, ax_dhyb):
        ln = ax.axvline(profile_pos, ls="--", lw=trace_lw, color="k", zorder=5)
        ln.set_path_effects(
            [pe.Stroke(linewidth=trace_outline_lw, foreground="white"), pe.Normal()]
        )

    ax_prof = fig2.add_subplot(gs2[1, :])
    y_kw = kw_mip[:, idx_profile]
    y_msgb = msgb_mip[:, idx_profile]
    y_hyb = hyb_mip[:, idx_profile]
    ax_prof.plot(ts_arr, y_kw, label="k-Wave", lw=2)
    ax_prof.plot(ts_arr, y_msgb, "--", label="MSGB", lw=2)
    ax_prof.plot(ts_arr, y_hyb, ":", label="Hybrid", lw=2.5)
    ax_prof.set_xlabel("t (s)")
    ax_prof.set_ylabel(r"$\max_z\, p_t(y_s)$")
    ax_prof.set_title(f"Temporal profile at $y_s$ = {profile_pos:.2e} m")
    ax_prof.legend(frameon=True, fancybox=True, shadow=True, loc="best")
    ax_prof.grid(True, alpha=0.3, ls=":")

    fig2.savefig(PLOT_DIR / f"{exp_name}_fig2_sensor.png", dpi=300, bbox_inches="tight")
    plt.close(fig2)

    # --- Figure 3: Orthogonal MIP views ---
    def orth_mips(vol):
        """Return XY, YZ, ZX MIP projections."""
        xy = np.max(vol, axis=2).T  # (Ny, Nx)
        yz = np.max(vol, axis=0).T  # (Nz, Ny)
        zx = np.max(vol, axis=1).T  # (Nz, Nx)
        return xy, yz, zx

    p0_xy, p0_yz, p0_zx = orth_mips(p0_real)
    err_xy, err_yz, err_zx = orth_mips(np.abs(recon_diff))

    fig3 = plt.figure(figsize=(14, 6))
    gs3 = fig3.add_gridspec(nrows=2, ncols=3, hspace=0.3, wspace=0.3)

    # Row 1: p0 projections
    ax_xy = fig3.add_subplot(gs3[0, 0])
    im_xy = ax_xy.imshow(p0_xy, origin="lower", cmap="hot")
    ax_xy.set_title(r"$p_0$ (XY)", fontsize=12)
    ax_xy.set_xlabel("X")
    ax_xy.set_ylabel("Y")
    ax_xy.set_xticks([])
    ax_xy.set_yticks([])
    add_colorbar(fig3, ax_xy, im_xy, size="4%", pad=0.05)

    ax_yz = fig3.add_subplot(gs3[0, 1])
    im_yz = ax_yz.imshow(p0_yz, origin="lower", cmap="hot")
    ax_yz.set_title(r"$p_0$ (YZ)", fontsize=12)
    ax_yz.set_xlabel("Y")
    ax_yz.set_ylabel("Z")
    ax_yz.set_xticks([])
    ax_yz.set_yticks([])
    add_colorbar(fig3, ax_yz, im_yz, size="4%", pad=0.05)

    ax_zx = fig3.add_subplot(gs3[0, 2])
    im_zx = ax_zx.imshow(p0_zx, origin="lower", cmap="hot")
    ax_zx.set_title(r"$p_0$ (ZX)", fontsize=12)
    ax_zx.set_xlabel("X")
    ax_zx.set_ylabel("Z")
    ax_zx.set_xticks([])
    ax_zx.set_yticks([])
    add_colorbar(fig3, ax_zx, im_zx, size="4%", pad=0.05)

    # Row 2: Error projections
    ax_exy = fig3.add_subplot(gs3[1, 0])
    im_exy = ax_exy.imshow(err_xy, origin="lower", cmap="hot")
    ax_exy.set_title(r"$|p_0^{\mathrm{recon}} - p_0|$ (XY)", fontsize=12)
    ax_exy.set_xlabel("X")
    ax_exy.set_ylabel("Y")
    ax_exy.set_xticks([])
    ax_exy.set_yticks([])
    add_colorbar(fig3, ax_exy, im_exy, size="4%", pad=0.05)

    ax_eyz = fig3.add_subplot(gs3[1, 1])
    im_eyz = ax_eyz.imshow(err_yz, origin="lower", cmap="hot")
    ax_eyz.set_title(r"$|p_0^{\mathrm{recon}} - p_0|$ (YZ)", fontsize=12)
    ax_eyz.set_xlabel("Y")
    ax_eyz.set_ylabel("Z")
    ax_eyz.set_xticks([])
    ax_eyz.set_yticks([])
    add_colorbar(fig3, ax_eyz, im_eyz, size="4%", pad=0.05)

    ax_ezx = fig3.add_subplot(gs3[1, 2])
    im_ezx = ax_ezx.imshow(err_zx, origin="lower", cmap="hot")
    ax_ezx.set_title(r"$|p_0^{\mathrm{recon}} - p_0|$ (ZX)", fontsize=12)
    ax_ezx.set_xlabel("X")
    ax_ezx.set_ylabel("Z")
    ax_ezx.set_xticks([])
    ax_ezx.set_yticks([])
    add_colorbar(fig3, ax_ezx, im_ezx, size="4%", pad=0.05)

    fig3.tight_layout()
    fig3.savefig(
        PLOT_DIR / f"{exp_name}_fig3_orthogonal.png", dpi=300, bbox_inches="tight"
    )
    plt.close(fig3)


# -----------------------------------------------------------------------------
# Forward experiment runner
# -----------------------------------------------------------------------------


def run_3d_forward_experiment(
    exp_name: str,
    p0: jnp.ndarray,
    domain: geometry.Domain,
    sensors: geometry.Sensor,
    ts: jnp.ndarray,
    wpt: MSWPT,
    dyadic_decomp: DyadicDecomposition,
    use_hybrid: bool = True,
):
    """
    Run k-Wave, MSGB, and optional Hybrid forward solvers and compute errors.
    """

    print(f"\n===== Experiment: {exp_name} =====")
    print(
        f"Domain N={domain.N}, dx={domain.dx}, cfl={domain.cfl}, periodic={domain.periodic}"
    )

    # MSWPT coefficients
    coeffs = wpt.forward(p0, input_type="spatial")

    def find_min_K_for_target_error(
        coeffs, p0, inv_wpt, tau=0.01, Kmin=128, Kmax=None, verbose=True
    ):
        """
        Binary search to find minimum K coefficients for target reconstruction error.
        """
        total = len(coeffs)
        if Kmax is None:
            Kmax = total

        Kmin = max(1, min(Kmin, Kmax))

        def select_topK_simple(K):
            """Simple top-K by magnitude."""
            K = min(K, total)
            abs_coeffs = jnp.abs(coeffs)

            if K < total:
                partition_idx = jnp.argpartition(abs_coeffs, total - K)[-K:]
            else:
                partition_idx = jnp.arange(total)

            sorted_idx = partition_idx[jnp.argsort(-abs_coeffs[partition_idx])]
            return sorted_idx, coeffs[sorted_idx]

        def test_K(K):
            """Test if K coefficients achieve target error."""
            indices, values = select_topK_simple(K)

            sparse_coeffs = jnp.zeros_like(coeffs)
            sparse_coeffs = sparse_coeffs.at[indices].set(values)
            recon = inv_wpt.inverse(sparse_coeffs, output_type="spatial")

            error = jnp.linalg.norm(p0 - recon.real) / (jnp.linalg.norm(p0) + 1e-30)
            return float(error)

        if verbose:
            print(f"\nSearching for minimum K to achieve {tau * 100:.1f}% error...")
            print(
                f"{'K':<10} {'Actual K':<10} {'Error %':<10} {'Target %':<10} {'Status':<10}"
            )
            print("-" * 60)

        # Binary search
        left, right = Kmin, Kmax
        best_K = Kmax

        while left <= right:
            mid = (left + right) // 2
            error = test_K(mid)

            if verbose:
                status = "✓ Pass" if error <= tau else "✗ Fail"
                print(
                    f"{mid:<10} {mid:<10} {error * 100:<10.2f} {tau * 100:<10.2f} {status:<10}"
                )

            if error <= tau:
                best_K = mid
                right = mid - 1
            else:
                left = mid + 1

        indices, values = select_topK_simple(best_K)

        if verbose:
            final_error = test_K(best_K)
            print("-" * 60)
            print(f"✓ Found minimum K = {best_K} (error = {final_error * 100:.2f}%)")

        return best_K, indices, values

    print(f"Total coefficients: {len(coeffs)}")

    # Find minimum K
    tau = 1.0
    K, indices, coeff_vals = find_min_K_for_target_error(
        coeffs=coeffs, p0=p0, inv_wpt=wpt, tau=tau, Kmin=1, Kmax=None, verbose=True
    )

    print(f"\n[result] K={K} coefficients needed for {tau * 100:.1f}% error")
    print(f"Selected coefficients: {len(indices)}")

    threshold = int(indices.size)
    print(f"[{exp_name}] Selected {threshold} coefficients")

    coeffs_array = wpt.convert_to_array(coeffs)
    p0_recon = utils.reconstruct_from_selection(
        coeffs, indices, coeff_vals, wpt, output_type="spatial"
    ).real

    # MSGB solver
    batch_size = 128
    sum_method = "scan_real"

    c_param = domain.c
    if callable(c_param):
        sample_points = jnp.stack(
            [
                jnp.zeros(domain.ndim),
                0.5 * domain.grid_size,
                jnp.array(domain.dx),
            ],
            axis=0,
        )
        c_vals = domain.c_fn(sample_points)
        is_homogeneous = bool(jnp.allclose(c_vals, c_vals[0]))
    else:
        c_arr = jnp.asarray(c_param)
        is_homogeneous = c_arr.ndim == 0 or c_arr.size == 1

    if is_homogeneous:
        ode_solver = gb_solvers.solve_hom_diag
        ode_config = None
    else:
        ode_solver = gb_solvers.solve_ODE_base
        ode_config = SolverConfig.from_precision(max_steps=4096, rtol=1e-4, atol=1e-6)

    if num_devices > 1:
        mesh = jax.make_mesh((num_devices,), ("x",))
        sharding = ShardingStrategy(mesh, beam_axis="x")
    else:
        sharding = None

    msgb_solver = MSGBSolver(
        thr=threshold,
        thr_strat="top_n",
        batch_size=batch_size,
        input_type="spatial",
        ode_solver=ode_solver,
        sum_method=sum_method,
        sharding=sharding,
        ode_config=ode_config,
    )

    print(f"[{exp_name}] Running MSGB solver...")
    t0 = time()
    gb_data = msgb_solver.forward(p0, domain, sensors, ts, wpt)[0].block_until_ready()
    t1 = time()
    print(f"[{exp_name}] MSGB runtime: {t1 - t0:.3f}s")
    gb_data = gb_data.real

    # k-Wave solver
    simulation_options = SimulationOptions(
        data_cast="double",
        smooth_p0=False,
        save_to_disk=True,
    )
    execution_options = SimulationExecutionOptions(
        is_gpu_simulation=is_gpu,
        delete_data=False,
        verbose_level=0,
        show_sim_log=False,
    )
    kwave_solver = KWaveSolver(simulation_options, execution_options)

    print(f"[{exp_name}] Running k-Wave solver...")
    t0 = time()
    sensor_data_kw = kwave_solver.forward(p0, domain, sensors.binary_mask, ts)
    t1 = time()
    print(f"[{exp_name}] k-Wave runtime: {t1 - t0:.3f}s")

    # Hybrid solver
    cutoff_freq = None
    if use_hybrid:
        box_corners = None
        cutoff_freq = 15.0  # low-frequency split

        hybrid_solver = HybridSolver(
            lf_solver=kwave_solver,
            hf_solver=msgb_solver,
            downsample=False,
            box_corners=box_corners,
            cutoff_freq=cutoff_freq,
            input_type="spatial",
            interp_method="fourier",
            dt_oversample=0,
            beta=12.0,
        )

        print(f"[{exp_name}] Running hybrid solver...")
        t0 = time()
        hybrid_data = hybrid_solver.forward(p0, domain, sensors, ts, wpt)
        t1 = time()
        print(f"[{exp_name}] Hybrid runtime: {t1 - t0:.3f}s")
    else:
        hybrid_data = None

    # sensor_data_kw = jnp.reshape(sensor_data_kw, jnp.shape(gb_data))
    print(
        f"[{exp_name}] k-Wave and MSGB data shapes: {sensor_data_kw.shape}, {gb_data.shape}"
    )
    # Metrics
    U_kw = np.asarray(sensor_data_kw)
    U_msgb = np.asarray(gb_data)
    print(f"[{exp_name}] k-Wave and MSGB data shapes: {U_kw.shape}, {U_msgb.shape}")
    metrics = {}

    E2_msgb = relative_L2(U_msgb, U_kw)
    Einf_msgb = relative_Linf(U_msgb, U_kw)
    Eeng_msgb = energy_error(U_msgb, U_kw)
    metrics.update(
        E2_msgb=E2_msgb,
        Einf_msgb=Einf_msgb,
        Eeng_msgb=Eeng_msgb,
        threshold=threshold,
    )

    print(f"[{exp_name}] MSGB errors vs k-Wave:")
    print(f"  E2       = {E2_msgb:.3e}")
    print(f"  Einf     = {Einf_msgb:.3e}")
    print(f"  E_energy = {Eeng_msgb:.3e}")

    if use_hybrid:
        U_hyb = jnp.reshape(hybrid_data, U_kw.shape)

        E2_hyb = relative_L2(U_hyb, U_kw)
        Einf_hyb = relative_Linf(U_hyb, U_kw)
        Eeng_hyb = energy_error(U_hyb, U_kw)
        metrics.update(
            E2_hyb=E2_hyb,
            Einf_hyb=Einf_hyb,
            Eeng_hyb=Eeng_hyb,
        )
        print(f"[{exp_name}] Hybrid errors vs k-Wave:")
        print(f"  E2       = {E2_hyb:.3e}")
        print(f"  Einf     = {Einf_hyb:.3e}")
        print(f"  E_energy = {Eeng_hyb:.3e}")

    # Plots
    plot_experiment_results(
        exp_name=exp_name,
        p0=p0,
        p0_recon=p0_recon,
        coeffs_array=coeffs_array,
        domain=domain,
        sensors=sensors,
        ts=ts,
        sensor_kw=sensor_data_kw,
        sensor_msgb=gb_data,
        sensor_hyb=hybrid_data,
        dyadic_decomp=dyadic_decomp,
        cutoff_freq=cutoff_freq,
    )

    # output_dir = DATA_DIR / "forward_3d_outputs"
    # output_dir.mkdir(exist_ok=True)
    # np.save(output_dir / f"{exp_name}_kwave.npy", U_kw)
    # np.save(output_dir / f"{exp_name}_msgb.npy", U_msgb)
    # if hybrid_data is not None:
    #     np.save(output_dir / f"{exp_name}_hybrid.npy", np.asarray(hybrid_data))

    return metrics, sensor_data_kw, gb_data, hybrid_data


# -----------------------------------------------------------------------------
# Main driver: define experiments and run them
# -----------------------------------------------------------------------------


def main():
    # Base domain & time grid for homogeneous sound speed
    domain_hom = geometry.Domain(N=N, dx=dx, c=c_hom, cfl=cfl, periodic=periodic)
    ts = domain_hom.generate_time_domain()

    # Planar sensors at z=0
    binary_mask = jnp.zeros(N)
    binary_mask = binary_mask.at[:, :, 0].set(1)
    sensors = geometry.Sensor(binary_mask=binary_mask, domain=domain_hom)

    experiments = []

    # Two packets, homogeneous
    experiments.append(("3d_two_packets_hom", make_p0_two_wavepackets, c_hom))

    # # Two packets, heterogeneous
    # experiments.append(("3d_two_packets_het", make_p0_two_wavepackets, c_het_gauss))

    # # Point source, homogeneous
    # experiments.append(("3d_point_hom", make_p0_point_source, c_hom))

    # # Point source, heterogeneous
    # experiments.append(("3d_point_het", make_p0_point_source, c_het_gauss))

    # p0_oab, c_grid_oab = make_p0_oabreast()
    # homogeneous c for OABreast
    # experiments.append(("3d_oabreast_hom", lambda: p0_oab, c_hom))
    # hetero c from grid for OABreast
    # c_oab_fn = utils.make_c_function_from_grid(c_grid_oab, spacing=dx, origin=(0.0, 0.0, 0.0))
    # experiments.append(("3d_oabreast_het", lambda: p0_oab, c_oab_fn))

    all_metrics = {}

    for name, p0_fn, c_fn in experiments:
        print("\n" + "=" * 70)
        print(f"Running experiment: {name}")
        print("=" * 70)

        # Build domain for this c(x)
        domain = geometry.Domain(N=N, dx=dx, c=c_fn, cfl=cfl, periodic=periodic)
        ts = domain.generate_time_domain()

        # Update sensor mask for this domain
        binary_mask = jnp.zeros(N)
        binary_mask = binary_mask.at[:, :, 0].set(1)
        sensors = geometry.Sensor(binary_mask=binary_mask, domain=domain)

        p0 = p0_fn()

        metrics, sensor_kw, sensor_msgb, sensor_hyb = run_3d_forward_experiment(
            exp_name=name,
            p0=p0,
            domain=domain,
            sensors=sensors,
            ts=ts,
            wpt=wpt,
            dyadic_decomp=dyadic_decomp,
            use_hybrid=True,
        )
        all_metrics[name] = metrics

    summary_path = DATA_DIR / "forward_3d_summary.csv"
    fieldnames = [
        "experiment",
        "E2_msgb",
        "Einf_msgb",
        "Eeng_msgb",
        "threshold",
        "E2_hyb",
        "Einf_hyb",
        "Eeng_hyb",
    ]
    with summary_path.open("a", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        for name, m in all_metrics.items():
            writer.writerow(
                dict(
                    experiment=name,
                    E2_msgb=m["E2_msgb"],
                    Einf_msgb=m["Einf_msgb"],
                    Eeng_msgb=m["Eeng_msgb"],
                    threshold=m["threshold"],
                    E2_hyb=m.get("E2_hyb"),
                    Einf_hyb=m.get("Einf_hyb"),
                    Eeng_hyb=m.get("Eeng_hyb"),
                )
            )
    print(f"\nSaved summary metrics to {summary_path}")

    # Print a compact summary table
    print("\n\n===== Summary of 3D experiments =====")
    for name, m in all_metrics.items():
        print(f"\n{name}:")
        print(
            f"  MSGB:   E2={m['E2_msgb']:.3e}, Einf={m['Einf_msgb']:.3e}, Eeng={m['Eeng_msgb']:.3e}, thr={m['threshold']}"
        )
        if "E2_hyb" in m:
            print(
                f"  Hybrid: E2={m['E2_hyb']:.3e}, Einf={m['Einf_hyb']:.3e}, Eeng={m['Eeng_hyb']:.3e}"
            )


if __name__ == "__main__":
    main()